In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#### Preprocessing

In [2]:
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]


train_df = pd.read_csv('adult.data', names=column_names, sep=', ', engine='python', na_values='?')

test_df = pd.read_csv('adult.test', names=column_names, sep=', ', engine='python',
                      skiprows=1, na_values='?')

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

label_enc = {"<=50K": 0, ">50K": 1, "<=50K.": 0, ">50K.": 1}
train_df['income'] = train_df['income'].map(label_enc)
test_df['income'] = test_df['income'].map(label_enc)

#### Train test split

In [3]:
X = train_df.drop('income', axis=1)
y = train_df['income']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_test = test_df.drop('income', axis=1)
y_test = test_df['income']

numeric_features = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
categorical_features = [col for col in X.columns if col not in numeric_features]

numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

In [4]:
import pickle
import os

os.makedirs('data/processed', exist_ok=True)

# Saving data
data_to_save = {
    # Dataframes
    'train_df': train_df,
    'test_df': test_df,
    
    # Processed data
    'X_train': X_train,
    'X_val': X_val,
    'X_test': X_test,
    'y_train': y_train,
    'y_val': y_val,
    'y_test': y_test,
    
    # Preprocessed versions
    'X_train_proc': X_train_proc,
    'X_val_proc': X_val_proc,
    'X_test_proc': X_test_proc,
    
    'preprocessor': preprocessor,
    
    # Feature names
    'numeric_features': numeric_features,
    'categorical_features': categorical_features
}

with open('data/processed/preprocessed_data.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)

print("Saved preprocessed data to data/processed/preprocessed_data.pkl")

Saved preprocessed data to data/processed/preprocessed_data.pkl
